In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np
tqdm.pandas()
import os

import fragnnet
import fragnnet.utils.data_utils as data_utils
from fragnnet.utils.data_utils import par_apply_series, par_apply_df_rows

In [2]:
df = pd.read_csv("../data/raw/massspecgym/MassSpecGym.tsv", sep="\t")

In [3]:
df.shape

(231104, 14)

In [4]:
df.columns

Index(['identifier', 'mzs', 'intensities', 'smiles', 'inchikey', 'formula',
       'precursor_formula', 'parent_mass', 'precursor_mz', 'adduct',
       'instrument_type', 'collision_energy', 'fold', 'simulation_challenge'],
      dtype='object')

In [5]:
df["inchikey"].iloc[0]

'VFMQMACUYWGDOJ'

In [6]:
sim_df = df.copy(deep=True)
sim_df = sim_df[sim_df["simulation_challenge"]]
sim_df = sim_df[["identifier","mzs","intensities","smiles","adduct","instrument_type","collision_energy","fold","precursor_mz","formula"]]

In [7]:
sim_df.shape

(119029, 10)

In [8]:
sim_df.columns

Index(['identifier', 'mzs', 'intensities', 'smiles', 'adduct',
       'instrument_type', 'collision_energy', 'fold', 'precursor_mz',
       'formula'],
      dtype='object')

In [9]:
sim_df["instrument_type"].value_counts()

instrument_type
Orbitrap    81323
QTOF        37706
Name: count, dtype: int64

In [10]:
sim_df["adduct"].value_counts()

adduct
[M+H]+    119029
Name: count, dtype: int64

In [11]:
sim_df["collision_energy"].value_counts()

collision_energy
20.000000     17910
60.000000     15869
30.000000     12584
10.000000      7042
45.000000      6232
              ...  
6.484782          1
11.494905         1
137.938860        1
22.989810         1
17.403515         1
Name: count, Length: 10117, dtype: int64

In [12]:
# spectrum stuff
spec_df = sim_df.copy(deep=True)
spec_df = spec_df.drop(columns=["fold"])
spec_df = spec_df.rename(columns={
    "identifier":"dset_spec_id",
    "collision_energy":"col_energy",
    "adduct":"prec_type",
    "precursor_mz":"prec_mz"
})
spec_df.loc[:,"inst_type"] = spec_df["instrument_type"].apply(lambda x: "FT" if x == "Orbitrap" else "QTOF")
spec_df.loc[:,"frag_mode"] = spec_df["instrument_type"].apply(lambda x: "HCD" if x == "Orbitrap" else "CID")
spec_df = spec_df.drop(columns=["instrument_type"])
spec_df.loc[:,"dset"] = "msg"
spec_df.loc[:,"spec_id"] = np.arange(spec_df.shape[0])
def parse_peaks(row):
    peaks = []
    for mz, intens in zip(row["mzs"].split(","),row["intensities"].split(",")):
        peaks.append((float(mz),999.*float(intens)))
    return peaks
spec_df.loc[:,"peaks"] = spec_df.progress_apply(parse_peaks, axis=1)
spec_df = spec_df.drop(columns=["mzs","intensities"])
assert not pd.to_numeric(spec_df["col_energy"], errors='coerce').isna().any()
spec_df.loc[:,"ace"] = pd.to_numeric(spec_df["col_energy"], errors='coerce')
spec_df = spec_df.drop(columns=["col_energy"])
# extra stuff
spec_df.loc[:,"nce"] = np.nan
spec_df.loc[:,"ri"] = np.nan
spec_df.loc[:,"ion_mode"] = "P"
spec_df.loc[:,"res"] = np.nan
spec_df.loc[:,"exact_mass"] = np.nan
spec_df.loc[:,"col_gas"] = np.nan
spec_df.loc[:,"spec_type"] = "MS2"
spec_df.loc[:,"inchikey"] = np.nan

  0%|          | 0/119029 [00:00<?, ?it/s]

100%|██████████| 119029/119029 [00:02<00:00, 43755.00it/s]


In [13]:
# molecule stuff
smiles_set = set(spec_df["smiles"])
print(len(smiles_set))
smiles_to_mid = {smiles:i for i,smiles in enumerate(sorted(smiles_set))}
spec_df.loc[:,"mol_id"] = spec_df["smiles"].map(smiles_to_mid)
spec_df = spec_df.drop(columns=["smiles"])
# group stuff
group_df = spec_df.drop(columns=["spec_id","dset_spec_id","peaks","nce","ace","res","prec_mz","ri","col_gas","formula","exact_mass","inchikey"]).drop_duplicates()
group_df.loc[:,"group_id"] = np.arange(group_df.shape[0])
spec_df = spec_df.merge(group_df,how="inner")

18110


In [14]:
# molecule dataframe
mol_df = pd.DataFrame(zip(sorted(smiles_set),list(range(len(smiles_set)))),columns=["smiles","mol_id"])
mol_df.loc[:,"mol"] = par_apply_series(mol_df["smiles"],data_utils.mol_from_smiles)
mol_df.loc[:,"inchikey_s"] = par_apply_series(mol_df["mol"],data_utils.mol_to_inchikey_s)
mol_df.loc[:,"scaffold"] = par_apply_series(mol_df["mol"],data_utils.get_murcko_scaffold)
mol_df.loc[:,"formula"] = par_apply_series(mol_df["mol"],data_utils.mol_to_formula)
mol_df.loc[:,"inchi"] = par_apply_series(mol_df["mol"],data_utils.mol_to_inchi)
mol_df.loc[:,"mw"] = par_apply_series(mol_df["mol"],lambda mol: data_utils.mol_to_mol_weight(mol,exact=False))
mol_df.loc[:,"exact_mw"] = par_apply_series(mol_df["mol"],lambda mol: data_utils.mol_to_mol_weight(mol,exact=True))
mol_df.loc[:,"num_atoms"] = par_apply_series(mol_df["mol"],data_utils.mol_to_num_atoms)
mol_df.loc[:,"num_bonds"] = par_apply_series(mol_df["mol"],data_utils.mol_to_num_bonds)
mol_df.loc[:,"charge"] = par_apply_series(mol_df["mol"],data_utils.mol_to_charge)
mol_df.loc[:,"single_mol"] = par_apply_series(mol_df["mol"],data_utils.check_single_mol)
mol_df.loc[:,"num_radicals"] = par_apply_series(mol_df["mol"],data_utils.mol_to_num_radicals)
assert not (mol_df["smiles"] == "").any()
assert not (mol_df["formula"] == "").any()
assert not (mol_df["exact_mw"] == 0).any()

<function mol_from_smiles at 0x7d19209d63b0>:   0%|          | 0/18110 [00:00<?, ?it/s]

<function mol_to_num_radicals at 0x7d19209d6950>: 100%|██████████| 18110/18110 [00:00<00:00, 49355.59it/s]


In [15]:
# remove invalid mols and corresponding spectra
all_mol_id = set(mol_df["mol_id"])
mol_df = mol_df.dropna(subset=["mol"],axis=0)
bad_mol_id = all_mol_id - set(mol_df["mol_id"])
print("> bad_mol_id",len(bad_mol_id))
spec_df = spec_df[~spec_df["mol_id"].isin(bad_mol_id)]

> bad_mol_id 0


In [16]:
# check how many formulae/inchikeys are different
formula_df = spec_df[["dset_spec_id","spec_id","mol_id","formula"]].copy()
formula_df = formula_df[formula_df["formula"] != ""]
formula_df = formula_df.merge(mol_df[["mol_id","formula"]],on="mol_id",how="inner")
diff_formula_df = formula_df[formula_df["formula_x"] != formula_df["formula_y"]]
print("> formula inconsistencies")
print(diff_formula_df["mol_id"].nunique(), diff_formula_df["spec_id"].nunique())
# inchikey_df = spec_df[["dset_spec_id","spec_id","mol_id","inchikey"]].copy().dropna()
# inchikey_df["inchikey_s"] = inchikey_df["inchikey"].str[:14]
# inchikey_df = inchikey_df.drop(columns=["inchikey"])
# inchikey_df = inchikey_df.merge(mol_df[["mol_id","inchikey_s"]],on="mol_id",how="inner")
# diff_inchikey_df = inchikey_df[inchikey_df["inchikey_s_x"] != inchikey_df["inchikey_s_y"]]
# print("> inchikey_s inconsistencies")
# print(diff_inchikey_df["mol_id"].nunique(), diff_inchikey_df["spec_id"].nunique())
# mass_df = spec_df[["dset_spec_id","spec_id","mol_id","exact_mass"]].copy().rename(columns={"exact_mass":"exact_mw"})
# # keep only non-trivial exact_mw annotations from the spec_df
# mass_df = mass_df[~mass_df["exact_mw"].isna() & mass_df["exact_mw"] > 0.0]
# # compare with reported exact_mw from the mol_df
# mass_df = mass_df.merge(mol_df[["mol_id","exact_mw"]],on="mol_id",how="inner")
# diff_mass_df = mass_df[(mass_df["exact_mw_x"] - mass_df["exact_mw_y"]).abs() > 0.1]
# print("> mass inconsistencies")
# print(diff_mass_df["mol_id"].nunique(), diff_mass_df["spec_id"].nunique())
# spec_df = spec_df.drop(columns=["formula","inchikey","exact_mass"])

> formula inconsistencies
3 10


In [17]:
spec_df = spec_df.drop(columns=["formula","inchikey","exact_mass"])

In [18]:
# display(diff_formula_df)
# print(mol_df[mol_df["mol_id"] == 14654]["smiles"].iloc[0])
# print(df[df["identifier"] == "MassSpecGymID0178193"]["smiles"].iloc[0])
# print(data_utils.mol_to_smiles(mol_df[mol_df["mol_id"] == 14654]["mol"].iloc[0]))
# display(sim_df[sim_df["identifier"].isin(diff_formula_df["dset_spec_id"])])

In [19]:
# fill in missing prec_mz by inferring them
spec_df = spec_df[~spec_df["spec_id"].isin(diff_formula_df["spec_id"])]
assert not (spec_df["prec_mz"].isna().any())

In [20]:
spec_df = spec_df.reset_index(drop=True)
mol_df = mol_df.reset_index(drop=True)

In [21]:
spec_df

,dset_spec_id,prec_type,prec_mz,inst_type,frag_mode,dset,spec_id,peaks,ace,nce,ri,ion_mode,res,col_gas,spec_type,mol_id,group_id
0,MassSpecGymID0000001,[M+H]+,288.12250,FT,HCD,msg,0,"[(91.0542, 245.0), (125.0233, 999.0), (154.049...",30.0,NaN,NaN,P,NaN,NaN,MS2,5422,0
1,MassSpecGymID0000002,[M+H]+,288.12250,FT,HCD,msg,1,"[(91.0542, 99.0), (125.0233, 281.0), (155.0577...",20.0,NaN,NaN,P,NaN,NaN,MS2,5422,0
2,MassSpecGymID0000003,[M+H]+,288.12250,FT,HCD,msg,2,"[(69.0343, 34.0), (91.0542, 314.0), (125.0233,...",40.0,NaN,NaN,P,NaN,NaN,MS2,5422,0
3,MassSpecGymID0000004,[M+H]+,288.12250,FT,HCD,msg,3,"[(69.0343, 179.0), (91.0542, 473.0), (110.06, ...",55.0,NaN,NaN,P,NaN,NaN,MS2,5422,0
4,MassSpecGymID0000005,[M+H]+,288.12250,FT,HCD,msg,4,"[(91.0542, 78.0), (125.0233, 184.0), (185.0961...",10.0,NaN,NaN,P,NaN,NaN,MS2,5422,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119014,MassSpecGymID0238972,[M+H]+,277.08592,FT,HCD,msg,119024,"[(43.017925, 12.387599999999999), (56.964737, ...",60.0,NaN,NaN,P,NaN,NaN,MS2,8607,20511
119015,MassSpecGymID0238973,[M+H]+,277.08592,FT,HCD,msg,119025,"[(79.021072, 26.71326), (231.081604, 27.95202)...",20.0,NaN,NaN,P,NaN,NaN,MS2,8607,20511
119016,MassSpecGymID0238974,[M+H]+,277.08592,FT,HCD,msg,119026,"[(43.017998, 58.12181999999999), (61.010777, 6...",45.0,NaN,NaN,P,NaN,NaN,MS2,8607,20511
119017,MassSpecGymID0239027,[M+H]+,457.10619,FT,HCD,msg,119027,"[(91.053917, 999.0), (298.098297, 184.17564)]",60.0,NaN,NaN,P,NaN,NaN,MS2,1743,20512


In [22]:
os.makedirs("../data/proc/massspecgym/",exist_ok=True)
spec_df.to_pickle("../data/proc/massspecgym/spec_df.pkl")
mol_df.to_pickle("../data/proc/massspecgym/mol_df.pkl")

In [23]:
# split
id_df = spec_df[["dset_spec_id","spec_id","group_id","mol_id"]].merge(
    df[["identifier","fold"]].rename(columns={"identifier":"dset_spec_id"}), 
    on="dset_spec_id",
    how="inner"
)
display(id_df["fold"].value_counts())
train_ids = id_df[id_df["fold"]=="train"][["spec_id","mol_id","group_id"]]
val_ids = id_df[id_df["fold"]=="val"][["spec_id","mol_id","group_id"]]
test_ids = id_df[id_df["fold"]=="test"][["spec_id","mol_id","group_id"]]


fold
train    99334
test      9954
val       9731
Name: count, dtype: int64

In [24]:
os.makedirs("../data/split/massspecgym/",exist_ok=True)
train_ids.to_csv("../data/split/massspecgym/train_ids.csv", index=False)
val_ids.to_csv("../data/split/massspecgym/val_ids.csv", index=False)
test_ids.to_csv("../data/split/massspecgym/test_ids.csv", index=False)


In [25]:
nist_spec_df = pd.read_pickle("../data/proc/nist20mona23v3/spec_df.pkl")
nist_mol_df = pd.read_pickle("../data/proc/nist20mona23v3/mol_df.pkl")

In [26]:
spec_df.columns

Index(['dset_spec_id', 'prec_type', 'prec_mz', 'inst_type', 'frag_mode',
       'dset', 'spec_id', 'peaks', 'ace', 'nce', 'ri', 'ion_mode', 'res',
       'col_gas', 'spec_type', 'mol_id', 'group_id'],
      dtype='object')

In [27]:
nist_spec_df.columns

Index(['spec_id', 'mol_id', 'prec_type', 'inst_type', 'frag_mode', 'spec_type',
       'ion_mode', 'dset', 'dset_spec_id', 'col_gas', 'res', 'ace', 'nce',
       'prec_mz', 'peaks', 'ri', 'group_id'],
      dtype='object')

In [28]:
set(spec_df.columns) - set(nist_spec_df.columns)

set()

In [29]:
set(nist_spec_df.columns) - set(spec_df.columns)

set()

In [30]:
nist_spec_df.iloc[0]["peaks"]

[(45.96, 46.35),
 (597.48, 51.85),
 (598.5, 41.36),
 (599.5, 669.33),
 (600.52, 40.76),
 (951.77, 51.85),
 (951.85, 40.96),
 (1437.29, 62.64),
 (1448.97, 75.02),
 (1449.97, 999.0),
 (1465.94, 52.35),
 (1466.0, 62.34),
 (1467.0, 704.29),
 (1467.98, 53.75),
 (1468.63, 40.86)]

In [31]:
spec_df.iloc[0]["peaks"]

[(91.0542, 245.0),
 (125.0233, 999.0),
 (154.0499, 80.0),
 (155.0577, 355.0),
 (185.0961, 349.0),
 (200.107, 45.0),
 (229.0859, 142.0),
 (246.1125, 734.0)]